# Proyecto final MEMAD – Semana 3
## Caso convexo: métodos de primer y segundo orden en regresión logística multiclase

**Objetivo de la semana:** estudiar un problema de optimización **convexo**, donde la teoría sí da
garantías, y comparar cuatro métodos con sus tasas de convergencia teóricas:

| Método | Orden | Información que usa | Convergencia teórica (local) |
|---|---|---|---|
| Descenso de gradiente (GD) | 1.º | Gradiente | Lineal, factor $1 - \mu/L$ |
| Gradiente conjugado no lineal (CG) | 1.º | Gradiente + direcciones conjugadas | Lineal, factor $\approx (\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)$ |
| L-BFGS | Cuasi-Newton | Aproximación de la curvatura con los últimos $m$ pasos | Lineal garantizada (BFGS completo: superlineal) |
| Newton | 2.º | Hessiana exacta | Cuadrática |

**Planteamiento:**
1. La ResNet-50 preentrenada en ImageNet se usa **congelada** como extractor de características
   (vector de 2 048 números por imagen).
2. Se reduce la dimensión con **PCA** (256 componentes) para que la Hessiana sea manejable.
3. Sobre esas características se entrena una **regresión logística multiclase con regularización L2**,
   que es un problema convexo (fuertemente convexo gracias a la regularización).

**Entregables que genera este notebook:** código de los cuatro métodos, gráficas de $\|\nabla f\|$ y
$f - f^*$ por iteración y por tiempo, y la comparación entre teoría y práctica.

**Cómo ejecutarlo:** GPU (T4 basta) → *Ejecutar todas*. La extracción de características tarda unos
minutos; la optimización corre en CPU en pocos minutos.

## 0. Configuración

In [ ]:
import os, time, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, models, transforms

DATA_DIR   = './brain_tumor_data'
OUT        = 'resultados_semana3'
PCA_DIM    = 256        # componentes principales
LAMBDA     = 1e-3       # regularización L2
VAL_FRAC   = 0.15
SPLIT_SEED = 42         # misma partición que en las semanas 1 y 2
GTOL       = 1e-10      # criterio de parada: ||∇f|| <= GTOL
TMAX       = 120        # tiempo máximo por método (s)
MAX_ITER   = {'GD': 5000, 'CG': 2000, 'L-BFGS': 2000, 'Newton': 50}
PESOS_IMAGENET = True   # ResNet-50 preentrenada en ImageNet

os.makedirs(OUT, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

## 1. Descarga del dataset

In [ ]:
if not os.path.isdir(os.path.join(DATA_DIR, 'Training')):
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print('No se pudieron leer los Secrets de Kaggle:', e)
    !pip install -q kaggle
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset --unzip -p {DATA_DIR}
print('Carpetas:', os.listdir(DATA_DIR))

## 2. Extracción de características con la ResNet-50 congelada

Se quita la última capa (`fc`) y cada imagen se convierte en un vector de 2 048 números. La red no se
entrena: solo se usa para transformar las imágenes. Las características se guardan en
`resultados_semana3/` para no repetir este paso.

In [ ]:
ruta_feat = os.path.join(OUT, 'caracteristicas_resnet50.npz')
if os.path.exists(ruta_feat):
    dat = np.load(ruta_feat)
    F_full, y_full, F_test, y_test = dat['F_full'], dat['y_full'], dat['F_test'], dat['y_test']
    CLASES = [str(c) for c in dat['clases']]
    print('Características leídas de', ruta_feat)
else:
    pesos = models.ResNet50_Weights.IMAGENET1K_V2 if PESOS_IMAGENET else None
    red = models.resnet50(weights=pesos)
    red.fc = nn.Identity()
    red = red.eval().to(device)
    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    @torch.no_grad()
    def extraer(carpeta):
        ds = datasets.ImageFolder(carpeta, transform=tf)
        dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)
        feats = []
        for x, _ in dl:
            with torch.autocast(device.type, enabled=device.type == 'cuda'):
                feats.append(red(x.to(device)).float().cpu())
        return torch.cat(feats).numpy(), np.array(ds.targets), ds.classes

    t0 = time.time()
    F_full, y_full, CLASES = extraer(os.path.join(DATA_DIR, 'Training'))
    F_test, y_test, _ = extraer(os.path.join(DATA_DIR, 'Testing'))
    np.savez_compressed(ruta_feat, F_full=F_full, y_full=y_full, F_test=F_test,
                        y_test=y_test, clases=np.array(CLASES))
    print(f'Extracción en {time.time() - t0:.0f} s')
print('Clases:', CLASES, '| Training:', F_full.shape, '| Testing:', F_test.shape)

## 3. Partición y PCA

Misma partición que en las semanas anteriores (15 % estratificado con semilla 42). El PCA se ajusta
**solo con el conjunto de entrenamiento**. Las componentes se escalan para que la primera tenga varianza 1,
**sin blanquear** las demás: así se conservan las diferencias de escala entre direcciones, que es lo que
hace al problema mal condicionado y vuelve interesante la comparación.

In [ ]:
np.random.seed(SPLIT_SEED)
tr_idx, va_idx = [], []
for c in np.unique(y_full):
    idx = np.where(y_full == c)[0]
    np.random.shuffle(idx)
    n_val = int(VAL_FRAC * len(idx))
    va_idx.extend(idx[:n_val]); tr_idx.extend(idx[n_val:])
tr_idx, va_idx = np.array(tr_idx), np.array(va_idx)

F_tr, y_tr = F_full[tr_idx].astype(np.float64), y_full[tr_idx]
F_va, y_va = F_full[va_idx].astype(np.float64), y_full[va_idx]
F_te, y_te = F_test.astype(np.float64), y_test

media = F_tr.mean(0)
_, S, Vt = np.linalg.svd(F_tr - media, full_matrices=False)
comp = Vt[:PCA_DIM]
var = S ** 2 / (len(F_tr) - 1)
escala = np.sqrt(var[0])

def proyectar(F):
    Z = (F - media) @ comp.T / escala
    return np.hstack([Z, np.ones((len(Z), 1))])      # columna de 1 para el sesgo

X_tr, X_va, X_te = proyectar(F_tr), proyectar(F_va), proyectar(F_te)
K = len(CLASES)
print(f'Varianza explicada por {PCA_DIM} componentes: {var[:PCA_DIM].sum() / var.sum() * 100:.1f} %')
print(f'Varianza de la componente 1 / componente {PCA_DIM}: {var[0] / var[PCA_DIM - 1]:.0f}')
print(f'X_tr: {X_tr.shape} | parámetros del modelo: {X_tr.shape[1] * K}')

## 4. El problema: regresión logística multiclase con regularización L2

Con $W \in \mathbb{R}^{d \times K}$, $p_i = \mathrm{softmax}(W^\top x_i)$ e $y_i$ codificado *one-hot*:

$$f(W) = \frac{1}{n}\sum_{i=1}^{n} \Big(\log \sum_{k} e^{w_k^\top x_i} - w_{y_i}^\top x_i\Big) + \frac{\lambda}{2}\|W\|_F^2$$

$$\nabla f(W) = \frac{1}{n} X^\top (P - Y) + \lambda W$$

$$\nabla^2 f = \frac{1}{n}\sum_{i} \big(\mathrm{diag}(p_i) - p_i p_i^\top\big) \otimes x_i x_i^\top + \lambda I$$

- $f$ es **fuertemente convexa** con parámetro $\mu \ge \lambda$ (la pérdida logística es convexa y el término L2 añade $\lambda I$).
- Su gradiente es **Lipschitz** con $L \le \tfrac12 \lambda_{\max}(X^\top X / n) + \lambda$, porque $\mathrm{diag}(p) - p p^\top \preceq \tfrac12 I$ (cota de Böhning).
- Por tanto, hay un **único mínimo** $W^*$ y la teoría de convergencia se aplica.

El vector de parámetros se ordena por clases: $w = (w_1, \dots, w_K)$, cada $w_k \in \mathbb{R}^d$.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(1, keepdims=True)

class LogisticaMulticlase:
    def __init__(self, X, y, K, lam):
        self.X, self.K, self.lam = X, K, lam
        self.n, self.d = X.shape
        self.Y = np.eye(K)[y]
        self.n_f = self.n_g = self.n_h = self.n_hv = 0      # contadores de evaluaciones

    def W(self, w):
        return w.reshape(self.K, self.d).T                  # d × K

    def f(self, w):
        self.n_f += 1
        W = self.W(w)
        Z = self.X @ W
        Z = Z - Z.max(1, keepdims=True)
        lse = np.log(np.exp(Z).sum(1))
        return (lse - (Z * self.Y).sum(1)).mean() + 0.5 * self.lam * w @ w

    def grad(self, w):
        self.n_g += 1
        W = self.W(w)
        P = softmax(self.X @ W)
        G = self.X.T @ (P - self.Y) / self.n + self.lam * W
        return G.T.ravel()

    def f_grad(self, w):
        return self.f(w), self.grad(w)

    def hess(self, w):
        # Hessiana exacta (dK × dK), por bloques de clases
        self.n_h += 1
        P = softmax(self.X @ self.W(w))
        d, K = self.d, self.K
        H = np.empty((d * K, d * K))
        for a in range(K):
            for b in range(a, K):
                c = P[:, a] * ((a == b) - P[:, b])
                blk = self.X.T @ (self.X * c[:, None]) / self.n
                H[a*d:(a+1)*d, b*d:(b+1)*d] = blk
                H[b*d:(b+1)*d, a*d:(a+1)*d] = blk.T
        H[np.diag_indices_from(H)] += self.lam
        return H

    def hvp(self, w, v):
        # Producto Hessiana-vector sin formar la Hessiana
        self.n_hv += 1
        P = softmax(self.X @ self.W(w))
        U = self.X @ self.W(v)
        R = P * U - P * (P * U).sum(1, keepdims=True)
        return (self.X.T @ R / self.n + self.lam * self.W(v)).T.ravel()

    def exactitud(self, w, X, y):
        return float(((X @ self.W(w)).argmax(1) == y).mean())

prob = LogisticaMulticlase(X_tr, y_tr, K, LAMBDA)
D = prob.d * K
L_cota = 0.5 * np.linalg.eigvalsh(X_tr.T @ X_tr / len(X_tr)).max() + LAMBDA
print(f'Dimensión del problema: {D} parámetros | L (cota) = {L_cota:.4f} | μ ≥ λ = {LAMBDA:g}')

### 4.1 Validación del gradiente y de la Hessiana (diferencias finitas)

Antes de optimizar se comprueba que las derivadas son correctas, comparándolas con diferencias
finitas centradas en un punto y una dirección aleatorios.

In [ ]:
rng = np.random.default_rng(0)
w_r, v_r = rng.normal(size=D) * 0.1, rng.normal(size=D)
eps = 1e-6
dd_f = (prob.f(w_r + eps * v_r) - prob.f(w_r - eps * v_r)) / (2 * eps)
dd_g = (prob.grad(w_r + eps * v_r) - prob.grad(w_r - eps * v_r)) / (2 * eps)
Hv_full, Hv_prod = prob.hess(w_r) @ v_r, prob.hvp(w_r, v_r)
err = lambda a, b: np.linalg.norm(a - b) / max(np.linalg.norm(b), 1e-300)
print(f'Gradiente  vs diferencias finitas: error relativo = {abs(prob.grad(w_r) @ v_r - dd_f) / abs(dd_f):.2e}')
print(f'Hessiana   vs diferencias finitas: error relativo = {err(Hv_full, dd_g):.2e}')
print(f'H·v (sin formar H) vs Hessiana:    error relativo = {err(Hv_prod, Hv_full):.2e}')

## 5. Implementación de los cuatro métodos

Todos parten de $w_0 = 0$ y se detienen cuando $\|\nabla f\| \le 10^{-10}$, al llegar al máximo de
iteraciones o al límite de tiempo. En cada iteración se registra $f$, $\|\nabla f\|$, el tiempo
acumulado y el iterado.

- **GD:** paso fijo $1/L$ (con la cota de $L$), que es el que usa la teoría.
- **CG no lineal (Polak–Ribière+):** búsqueda lineal casi exacta (Newton en una dimensión con productos
  Hessiana-vector), reinicio cada $D$ iteraciones o si la dirección no es de descenso.
- **L-BFGS:** memoria $m = 10$, recursión de dos lazos y búsqueda lineal con retroceso (Armijo).
- **Newton:** Hessiana exacta, sistema $H p = -\nabla f$ resuelto por Cholesky y paso amortiguado con
  Armijo (el paso completo $\alpha = 1$ se acepta cerca del óptimo).

In [ ]:
class Registro:
    def __init__(self, nombre):
        self.nombre, self.t0 = nombre, time.perf_counter()
        self.f, self.g, self.t, self.w = [], [], [], []
    def add(self, w, f, g):
        self.f.append(f); self.g.append(np.linalg.norm(g))
        self.t.append(time.perf_counter() - self.t0); self.w.append(w.copy())
    def parar(self, max_iter):
        return self.g[-1] <= GTOL or self.t[-1] > TMAX or len(self.f) > max_iter

def armijo(prob, w, f0, g0, p, a0=1.0, c=1e-4, rho=0.5):
    gp, a = g0 @ p, a0
    for _ in range(60):
        f1 = prob.f(w + a * p)
        if f1 <= f0 + c * a * gp:
            return a, f1
        a *= rho
    return a, f1

def gd(prob, w0, L):
    R, w = Registro('GD'), w0.copy()
    f, g = prob.f_grad(w); R.add(w, f, g)
    while not R.parar(MAX_ITER['GD']):
        w = w - g / L
        f, g = prob.f_grad(w); R.add(w, f, g)
    return R

def busqueda_exacta(prob, w, d, g0, iters=10):
    # min_a f(w + a d) por Newton en 1D: a <- a - φ'(a)/φ''(a)
    a, dphi = 0.0, g0 @ d
    for _ in range(iters):
        wa = w + a * d
        if a != 0.0:
            dphi = prob.grad(wa) @ d
        paso = dphi / (d @ prob.hvp(wa, d))
        a -= paso
        if abs(paso) <= 1e-12 * (1 + abs(a)):
            break
    return a

def cg(prob, w0):
    R, w = Registro('CG'), w0.copy()
    f, g = prob.f_grad(w); R.add(w, f, g)
    d, k = -g, 0
    while not R.parar(MAX_ITER['CG']):
        if g @ d >= 0:
            d = -g                                     # reinicio: no es dirección de descenso
        a = busqueda_exacta(prob, w, d, g)
        f_new = prob.f(w + a * d)
        if not (a > 0 and f_new <= f + 1e-4 * a * (g @ d)):
            a, f_new = armijo(prob, w, f, g, d)        # salvaguarda
        w = w + a * d
        g_new = prob.grad(w)
        beta = max(0.0, g_new @ (g_new - g) / (g @ g))  # Polak–Ribière+
        k += 1
        if k % D == 0:
            beta = 0.0
        d = -g_new + beta * d
        f, g = f_new, g_new
        R.add(w, f, g)
    return R

def lbfgs(prob, w0, m=10):
    R, w = Registro('L-BFGS'), w0.copy()
    f, g = prob.f_grad(w); R.add(w, f, g)
    S, Y = [], []
    while not R.parar(MAX_ITER['L-BFGS']):
        q, alfas = g.copy(), []
        for s, y in zip(reversed(S), reversed(Y)):
            a = (s @ q) / (y @ s); q -= a * y; alfas.append(a)
        gamma = (S[-1] @ Y[-1]) / (Y[-1] @ Y[-1]) if S else 1.0 / max(1.0, np.linalg.norm(g))
        r = gamma * q
        for (s, y), a in zip(zip(S, Y), reversed(alfas)):
            b = (y @ r) / (y @ s); r += s * (a - b)
        p = -r
        a, f_new = armijo(prob, w, f, g, p)
        w_new = w + a * p
        g_new = prob.grad(w_new)
        s, y = w_new - w, g_new - g
        if s @ y > 1e-12:
            S.append(s); Y.append(y)
            if len(S) > m:
                S.pop(0); Y.pop(0)
        w, f, g = w_new, f_new, g_new
        R.add(w, f, g)
    return R

def newton(prob, w0):
    R, w = Registro('Newton'), w0.copy()
    f, g = prob.f_grad(w); R.add(w, f, g)
    while not R.parar(MAX_ITER['Newton']):
        H = prob.hess(w)
        Lc = np.linalg.cholesky(H)                      # H es definida positiva
        p = -np.linalg.solve(Lc.T, np.linalg.solve(Lc, g))
        a, f = armijo(prob, w, f, g, p)
        w = w + a * p
        g = prob.grad(w)
        R.add(w, f, g)
    return R

## 6. Ejecución

In [ ]:
w0 = np.zeros(D)
registros, evals = {}, {}
for nombre, metodo in [('GD', lambda: gd(prob, w0, L_cota)), ('CG', lambda: cg(prob, w0)),
                       ('L-BFGS', lambda: lbfgs(prob, w0)), ('Newton', lambda: newton(prob, w0))]:
    prob.n_f = prob.n_g = prob.n_h = prob.n_hv = 0
    R = metodo()
    registros[nombre] = R
    evals[nombre] = dict(f=prob.n_f, grad=prob.n_g, hess=prob.n_h, hvp=prob.n_hv)
    print(f'{nombre:7s} iteraciones = {len(R.f) - 1:5d} | ||∇f|| final = {R.g[-1]:.2e} | '
          f'tiempo = {R.t[-1]:.2f} s')

## 7. Validación contra SciPy

Se resuelve el mismo problema con `scipy.optimize.minimize` (L-BFGS-B) como referencia independiente.

Para el óptimo de referencia $w^*$ se toma el iterado final con **menor norma del gradiente** y se refina
con dos pasos de Newton puros. Por convexidad fuerte, $\|w - w^*\| \le \|\nabla f(w)\|/\mu$, así que
un gradiente casi nulo garantiza un $w^*$ muy preciso. El valor de $f$ no sirve para esto, porque cerca del
mínimo es casi plano: $f - f^* \approx \tfrac{\mu}{2}\|w - w^*\|^2$.

In [ ]:
from scipy.optimize import minimize
ref = minimize(lambda w: prob.f_grad(w), w0, jac=True, method='L-BFGS-B',
               options=dict(maxiter=20000, gtol=1e-12, ftol=1e-16))
mejor = min(registros, key=lambda k: registros[k].g[-1])
w_star = registros[mejor].w[-1].copy()
for _ in range(2):                                     # refinamiento con Newton puro
    w_star = w_star - np.linalg.solve(prob.hess(w_star), prob.grad(w_star))
f_star = prob.f(w_star)
print(f'w* refinado a partir de {mejor}: ||∇f(w*)|| = {np.linalg.norm(prob.grad(w_star)):.2e}')
print(f'f* (nuestro)         = {f_star:.15f}')
print(f'f  (SciPy L-BFGS-B)  = {ref.fun:.15f}   |diferencia| = {abs(ref.fun - f_star):.2e}')
for nombre, R in registros.items():
    print(f'{nombre:7s} f − f* = {R.f[-1] - f_star:.2e} | ||w − w*|| = {np.linalg.norm(R.w[-1] - w_star):.2e}')

## 8. Curvas de convergencia: $f - f^*$ y $\|\nabla f\|$ por iteración y por tiempo

In [ ]:
COL = {'GD': '#1f77b4', 'CG': '#2ca02c', 'L-BFGS': '#ff7f0e', 'Newton': '#d62728'}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for nombre, R in registros.items():
    it = np.arange(len(R.f))
    gap = np.maximum(np.array(R.f) - f_star, 1e-16)
    kw = dict(color=COL[nombre], label=nombre, lw=1.8, marker='o' if len(it) < 60 else None, ms=3)
    axes[0, 0].loglog(it + 1, gap, **kw)
    axes[0, 1].loglog(it + 1, R.g, **kw)
    axes[1, 0].semilogy(R.t, gap, **kw)
    axes[1, 1].semilogy(R.t, R.g, **kw)
for ax, (tit, xl) in zip(axes.ravel(), [('f − f* por iteración', 'Iteración + 1 (log)'),
                                        ('||∇f|| por iteración', 'Iteración + 1 (log)'),
                                        ('f − f* por tiempo', 'Tiempo (s)'),
                                        ('||∇f|| por tiempo', 'Tiempo (s)')]):
    ax.set_title(tit); ax.set_xlabel(xl); ax.grid(alpha=0.3, which='both'); ax.legend()
fig.suptitle(f'Regresión logística multiclase (D = {D}, λ = {LAMBDA:g}): convergencia de los cuatro métodos')
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'convergencia.png'), dpi=130)
plt.show()

## 9. Teoría frente a práctica: tasas de convergencia

Se mide el error $e_k = \|w_k - w^*\|$ en escala logarítmica:

- **Convergencia lineal** ($e_{k+1} \approx \rho\, e_k$) se ve como una **recta**, de pendiente $\log \rho$.
  La tasa empírica $\rho$ se estima ajustando una recta a la segunda mitad de las iteraciones, antes del
  límite de redondeo.
- **Convergencia cuadrática** ($e_{k+1} \approx C e_k^2$): la curva cae cada vez más rápido y el número de
  cifras correctas se duplica en cada paso, así que $e_{k+1}/e_k^2$ se mantiene acotado.
- Las rectas teóricas parten del error inicial; lo que se compara es su **pendiente**, ya que al principio
  GD avanza más rápido en las direcciones bien condicionadas.
- **L-BFGS:** la teoría solo garantiza convergencia lineal (Liu y Nocedal, 1989). Con la Hessiana completa
  (BFGS) sería superlineal, así que interesa ver cuánto se acerca en la práctica.

Para GD con paso $1/L$, la teoría predice cerca del óptimo un factor $\rho = 1 - \mu/L$ por iteración en
$e_k$, donde $\mu = \lambda_{\min}(\nabla^2 f(w^*))$. Para CG se usa como referencia el factor de los
problemas cuadráticos, $(\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)$, con $\kappa = \lambda_{\max}/\lambda_{\min}$ de $\nabla^2 f(w^*)$.

In [ ]:
H_star = prob.hess(w_star)
ev = np.linalg.eigvalsh(H_star)
mu, L_star = ev[0], ev[-1]
kappa = L_star / mu
rho_gd = 1 - mu / L_cota
rho_cg = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)
print(f'Hessiana en w*: μ = {mu:.3e}, L = {L_star:.3e}, κ = {kappa:.0f}  (L usada por GD: {L_cota:.3e})')

def errores(R):
    return np.array([np.linalg.norm(w - w_star) for w in R.w])

def tasa_empirica(e):
    # pendiente de log(e) en la segunda mitad de las iteraciones por encima del límite de redondeo
    k = np.where(e > 1e-9 * e[0])[0]
    k = k[len(k) // 2:]
    if len(k) < 3:
        return float('nan')
    return float(np.exp(np.polyfit(k, np.log(e[k]), 1)[0]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tasas = {}
for nombre, R in registros.items():
    e = errores(R)
    tasas[nombre] = tasa_empirica(e)
    etiqueta = ('Newton (cuadrática: ver tabla)' if nombre == 'Newton'
                else f'{nombre} (ρ empírica = {tasas[nombre]:.4f})')
    for ax in axes:
        ax.semilogy(np.arange(len(e)), np.maximum(e, 1e-16), color=COL[nombre], lw=1.8,
                    marker='o' if len(e) < 60 else None, ms=3,
                    label=etiqueta)
# rectas teóricas, ancladas en el error inicial
k_gd = np.arange(len(registros['GD'].w))
e0 = errores(registros['GD'])[0]
axes[0].semilogy(k_gd, e0 * rho_gd ** k_gd, 'k--', lw=1, label=f'Teoría GD: ρ = 1 − μ/L = {rho_gd:.4f}')
k_cg = np.arange(len(registros['CG'].w))
for ax in axes:
    ax.semilogy(k_cg, e0 * rho_cg ** k_cg, ':', color=COL['CG'], lw=1.5,
                label=f'Referencia CG: ρ = (√κ−1)/(√κ+1) = {rho_cg:.4f}')
axes[1].set_xlim(0, max(len(registros[n].w) for n in ('CG', 'L-BFGS', 'Newton')) + 2)
for ax, tit in zip(axes, ['Todas las iteraciones', 'Detalle: primeras iteraciones']):
    ax.set_ylim(1e-16, e0 * 10); ax.set_xlabel('Iteración k'); ax.set_ylabel('||w(k) − w*||  (log)')
    ax.set_title(tit); ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=8)
fig.suptitle('Error ||w(k) − w*||: una recta en escala log indica convergencia lineal')
fig.tight_layout(); fig.savefig(os.path.join(OUT, 'tasas_convergencia.png'), dpi=130); plt.show()

print(f'\nTasa lineal GD:  empírica = {tasas["GD"]:.4f}  | teórica 1 − μ/L = {rho_gd:.4f}')
print(f'Tasa lineal CG:  empírica = {tasas["CG"]:.4f}  | referencia (√κ−1)/(√κ+1) = {rho_cg:.4f}')
print(f'Tasa L-BFGS:     empírica = {tasas["L-BFGS"]:.4f}')

# Newton: convergencia cuadrática
e = errores(registros['Newton'])
filas = []
for k in range(len(e)):
    filas.append({'k': k, '||∇f||': f'{registros["Newton"].g[k]:.2e}', 'e(k) = ||w(k) − w*||': f'{e[k]:.2e}',
                  'e(k)/e(k−1)': f'{e[k] / e[k-1]:.2e}' if k and e[k-1] > 0 else '—',
                  'e(k)/e(k−1)²': f'{e[k] / e[k-1]**2:.2e}' if k and e[k-1] > 1e-300 else '—'})
tabla_newton = pd.DataFrame(filas)
tabla_newton.to_csv(os.path.join(OUT, 'newton_iteraciones.csv'), index=False)
print('\nNewton, iteración por iteración:')
display(tabla_newton)
json.dump(dict(mu=mu, L_hess=L_star, L_cota=L_cota, kappa=kappa, rho_gd_teo=rho_gd, rho_cg_ref=rho_cg,
               tasas_empiricas=tasas), open(os.path.join(OUT, 'teoria.json'), 'w'), indent=1)

## 10. Tabla resumen

La exactitud es la del clasificador lineal resultante. Como el problema es convexo, todos los métodos
que convergen llegan al **mismo** modelo y, por tanto, a la misma exactitud: lo que cambia es el costo
de llegar.

In [ ]:
def iters_hasta(R, tol):
    gap = np.array(R.f) - f_star
    idx = np.where(gap <= tol)[0]
    return int(idx[0]) if len(idx) else None

filas = []
for nombre, R in registros.items():
    n_it = len(R.f) - 1
    ev_ = evals[nombre]
    filas.append({
        'Método': nombre,
        'Iteraciones': n_it,
        'Evaluaciones f / ∇f / H / Hv': f"{ev_['f']} / {ev_['grad']} / {ev_['hess']} / {ev_['hvp']}",
        'Tiempo total (s)': round(R.t[-1], 2),
        'Tiempo/iter. (ms)': round(1000 * R.t[-1] / max(n_it, 1), 2),
        'Iter. hasta f−f* ≤ 1e-6': iters_hasta(R, 1e-6) if iters_hasta(R, 1e-6) is not None else 'no alcanza',
        'Iter. hasta f−f* ≤ 1e-12': iters_hasta(R, 1e-12) if iters_hasta(R, 1e-12) is not None else 'no alcanza',
        '||∇f|| final': f'{R.g[-1]:.1e}',
        'f − f* final': f'{max(R.f[-1] - f_star, 0):.1e}',
        'Val. acc. (%)': round(100 * prob.exactitud(R.w[-1], X_va, y_va), 2),
        'Test acc. (%)': round(100 * prob.exactitud(R.w[-1], X_te, y_te), 2),
    })
tabla = pd.DataFrame(filas)
tabla.to_csv(os.path.join(OUT, 'tabla_metodos.csv'), index=False)
display(tabla)

# Historial completo para el informe
pd.concat([pd.DataFrame({'metodo': n, 'iter': np.arange(len(R.f)), 'tiempo': R.t, 'f': R.f,
                         'f_menos_fstar': np.array(R.f) - f_star, 'norma_grad': R.g,
                         'error_w': errores(R)}) for n, R in registros.items()]
          ).to_csv(os.path.join(OUT, 'historial.csv'), index=False)

## 11. Descargar resultados

Descarga un `.zip` con las figuras y las tablas para subirlo a Classroom (sin las características,
que pesan mucho).

In [ ]:
!cd {OUT} && zip -q ../semana3.zip *.png *.csv *.json
from google.colab import files
files.download('semana3.zip')